# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane:** Growth / Recovery / Momentum Prediction. **Question:** which pages should a senior SEO specialist review first, because they show early signs of moving?

This notebook is the synthesis. It rebuilds the D1 cohort, the gate and the shipped model from scratch, re-derives the headline comparison against the baseline, and produces the figures the paper embeds. The deeper audits live in the notebooks that ran them and are cited in place — §3's leakage hunt in `w03_feature_leakage_check`, the null simulation in `w04_signal_audit`, the forward walks in `w06_validation_audit`.

> Every number in prose below is printed by a cell in this notebook or one of the seven it synthesises. `work/tools/check_claims.py` enforces that by pooling the stdout of every executed cell across `work/notebooks/`.

## 1. Question

**The decision this supports.** Which pages a senior SEO specialist opens first, because they show early signs of decline, recovery, or unexpected momentum. The unit of analysis is one content item evaluated at a fixed decision-point date — not a day, not a client. The output is a ranked, explained flag list: a model score per page doubles as the sort key for the review queue. The model never acts. It flags a page with the reasons it was flagged; a specialist reads those and accepts or declines.

**The cost of a wrong call is asymmetric.** A false flag spends review time on a page that did not need it — recoverable. A missed signal means a real decline or recovery is never surfaced and the window to act passes — not recoverable. So recall matters here, not only precision at the top of the list.

**Why ML rather than a fixed rule.** A single threshold cannot separate a real 20% drop on a high-traffic page from meaningless noise on a page that went from 2 impressions to 1 — `trend_pct` in the starter data reaches 44,900%, produced entirely by tiny denominators. Weighing percentage change together with prior volume, position, age and freshness is the part a hand-written rule does badly.

Even *defining* "already declining" is design-sensitive: a 30-vs-30-day window and a 45-vs-45-day window disagree on **22.3%** of pages at a −20% cut, and order them at only Spearman **+0.565**.

*Source: `w01_research_question.ipynb`, `w02_ml_task_framing.ipynb`.*

## 2. Data

**Source.** The gated `FlyRank/internship-warehouse` release, build `v20260703`: `fact_content_daily_performance` (daily grain, aggregated to one row per page), `dim_content`, `dim_clients`. The 30k-row starter CSV has no daily granularity and cannot express a future window at all. The access token is supplied at runtime from a gitignored `.env` — never written into a cell, because this repo is public.

**Deliberately excluded, and why.**

| Excluded | Reason |
|---|---|
| `future_change_pct`, `future_impressions`, the three target columns | The label, or the window it is computed from |
| All of `fact_content_query_90d` | Its fixed 90-day window overlaps this lane's label window — any column from it leaks the future |
| `last_optimized_date`, `optimization_eligible_date` | 87.8% missing; naming suggests they populate only when FlyRank's own system acted on a page — the product-decision-as-feature trap |
| `provider_used`, `model_used` | Marked "not a model feature" in the data dictionary |
| `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | 100.0% zero across every content item in the window — zero variance |
| `client_hash_id`, `content_hash_id`, `keyword_hash_id`, `url_hash_id` | Pseudonyms — grouping, joining and splitting only, never features |
| `is_active`, `has_gsc_access`, `has_ga4_access`, `access_profile` | Client-level: constant inside a client, and the queue ranks pages *within* a client, so a client-constant cannot reorder anything |

**Timeline verified.** Every feature is drawn from 2025-12-31 → 2026-03-30; the label window opens 2026-03-31; no overlap. Three explicit leakage attacks behave as they should — injecting the label-generating ratio drives AUC to **0.92**, injecting a raw future count leaks weakly, and a random row split inflates the score by **+0.091** AUC over a client-grouped one, on the same ten seeds.

**A leak the leakage hunt missed.** `dim_content` is an **export-time snapshot**, not a point-in-time record: `content_updated_date` falls after the decision point for the large majority of cohort pages, and a single bulk export date accounts for roughly two fifths of all content items. `days_since_last_update` was therefore leaky and has been removed. The original hunt verified the timeline for the *fact table* windows and never asked the same question of the dimension table. A related consequence: an `is_deleted`/`is_published` filter added during that hunt was itself selection on the outcome window — judging a March decision by July status — and has been reverted.

**Client-identifying content:** none. The only identifiers that appear are pseudonymous `content_*` / `client_*` hashes, which `DATA_USE.md` permits for grouping, shown to demonstrate grain rather than as a row-level dump.

*Source: `w03_data_contract.ipynb`, `w03_feature_leakage_check.ipynb`, `w06_validation_audit.ipynb`.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**The target is one continuous number.** Three binary targets were replaced by a single signed difference:

```
target = asinh(future_daily_rate) - asinh(baseline_daily_rate)
```

Sign is direction, magnitude is size — a 21% dip and a 95% collapse are no longer the same label. `asinh(x) = log(x + sqrt(x^2 + 1))` is 0 at zero and converges to `log(2x)` for large x, so it behaves like a log ratio for healthy pages while staying finite when one reaches zero — which a ratio cannot represent at all, and that is **7.4%** of the D1 cohort. Where both are defined it agrees with the log ratio at Spearman **+0.9546**. It also deflates percentage swings at trivial volume: the same 50% drop is **−0.6931** by log ratio at every size, but **−0.0499** at 0.1 impressions/day against **−0.6931** at 100. Given a cohort whose median dead page ran 0.13 impressions a day, that deflation is the point.

**Windows.** Features from the trailing 90 days before the decision point; "already declining" from a 30-vs-30-day split of that window, matching FlyRank's own `trend_pct` convention rather than an invented one; label from the 30 days after. The future compares against the **recent 30-day** daily rate, not a 90-day average — using the 90-day average hides real recoveries behind stale history.

**Features.** Twenty-five were engineered; the shipped model uses **four**: `peak_ratio`, `prior_trend`, `impr_90d` and `avg_position`. `content_age_days` was removed by ML-09's ablation — it scored **−0.0645** on permutation importance, meaning shuffling it *improved* held-out fit. All four survivors are window aggregates, each computed by summing then dividing, never by averaging per-day rates. `peak_ratio` holds **0.6034** permutation importance against ≤0.0073 for every other feature, so this is effectively a one-feature model wearing four.

**Position is zero-based.** `gsc_avg_position` follows GSC's bulk-export convention where **0 is the top rank**, so average position is `SUM(sum_position)/SUM(impressions) + 1`. The starter CSV uses the opposite rule — `0` means *missing* — and applying that here filters out each page's best days for **53.4%** of the cohort.

**Missing values: flag first, then fill.** `has_*` indicators are computed before any fill, so "unknown" and "genuinely zero" stay distinguishable. Missingness follows `content_type`, so a median fill would stamp the content type into the feature — the model would appear to use search volume while actually using content type.

**The baseline.** A rule reads three fields only: age ≥ 180 days, impressions at or above the client's own median, and `slip <= 0.5` (a page that already lost half is not preventable); it ranks by age. On the same ten splits the gate lifts the decline rate from the cohort's **0.4228** to **0.5245**, but ranking by age inside it scores P@100 **0.5041** against a random-order bar of **0.5432** — **−0.0391 below random**. The gate is worth having; the ranking is not. So the number to beat is the gate with pages in arbitrary order, not the rule.

In [4]:
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

SEED = 8

# Check if running in Google Colab and use user secrets
if 'google.colab' in str(get_ipython()):
    from google.colab import userdata
    try:
        token = userdata.get('HF_TOKEN')
    except userdata.SecretNotFoundError:
        raise RuntimeError('Set HF_TOKEN in Colab secrets before running this notebook.')
else:
    token = os.environ.get('HF_TOKEN')
    if not token:
        raise RuntimeError('Set HF_TOKEN in the environment before running this notebook.')

REPO = 'FlyRank/internship-warehouse'
con = duckdb.connect()
# Authenticate remote Hugging Face reads without downloading dataset files locally.
safe_token = token.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")
MONTHS = ['2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
daily_paths = ', '.join(
    f"'hf://datasets/{REPO}/fact_content_daily_performance/month={month}/data_0.parquet'"
    for month in MONTHS
)
REL = f'read_parquet([{daily_paths}])'
DIM_REL = f"read_parquet('hf://datasets/{REPO}/dim_content.parquet')"
D1 = '2026-03-31'


def build_cohort(dstr, window_days=30):
    """The D1 cohort at an arbitrary decision date and outcome window.

    Copied verbatim from w07_action_playbook.ipynb, which copied it from
    w05_model.ipynb. Rebuilding it here rather than trusting a cached frame is
    what lets the prints below check this notebook against ML-08 and ML-10.
    """
    q = f"""
    WITH prior AS (
      SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) AS impr_90d,
        SUM(gsc_clicks) AS clicks_90d,
        SUM(gsc_sum_position) AS sum_position_90d,
        SUM(gsc_impressions) FILTER (
            WHERE report_date < DATE '{dstr}' - INTERVAL 30 DAY) AS older60_impr,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{dstr}' - INTERVAL 30 DAY) AS recent30_impr,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{dstr}' - INTERVAL 60 DAY
              AND report_date <  DATE '{dstr}' - INTERVAL 30 DAY) AS base30_impr
      FROM {REL}
      WHERE report_date >= DATE '{dstr}' - INTERVAL 90 DAY AND report_date < DATE '{dstr}'
      GROUP BY content_hash_id HAVING SUM(gsc_impressions) > 0),
    fut AS (
      SELECT content_hash_id, SUM(gsc_impressions) AS future_impr FROM {REL}
      WHERE report_date >= DATE '{dstr}'
        AND report_date < DATE '{dstr}' + INTERVAL {int(window_days)} DAY
      GROUP BY content_hash_id)
    SELECT p.*, COALESCE(f.future_impr, 0) AS future_impr
    FROM prior p LEFT JOIN fut f USING (content_hash_id)
    ORDER BY p.content_hash_id"""
    x = con.sql(q).df()
    for c in ['older60_impr', 'recent30_impr', 'base30_impr']:
        x[c] = x[c].fillna(0)
    dd = pd.Timestamp(dstr)
    x = x.merge(con.sql(f"SELECT content_hash_id, content_created_date FROM {DIM_REL}").df(),
                on='content_hash_id', how='left')
    x['baseline_daily'] = x['older60_impr'] / 60
    x['recent_daily'] = x['recent30_impr'] / 30
    x['future_daily'] = x['future_impr'] / window_days
    x['target'] = np.arcsinh(x['future_daily']) - np.arcsinh(x['baseline_daily'])
    x['declined'] = x['target'] < 0
    x['avg_position'] = x['sum_position_90d'] / x['impr_90d'].replace(0, np.nan) + 1
    x['content_age_days'] = (dd - pd.to_datetime(x['content_created_date'])).dt.days
    x['slip'] = np.where(x['baseline_daily'] > 0,
                         (x['baseline_daily'] - x['recent_daily']) / x['baseline_daily'], np.nan)
    x['peak_ratio'] = np.where(x['impr_90d'] > 0,
                               x['recent_daily'] / (x['impr_90d'] / 90), np.nan)
    x['prior_trend'] = np.where(x['base30_impr'] > 0,
                                (x['recent30_impr'] - x['base30_impr']) / x['base30_impr'], np.nan)
    med = x.groupby('client_hash_id')['impr_90d'].transform('median')
    gate = ((x['baseline_daily'] > 0) & (x['content_age_days'] >= 180)
            & (x['impr_90d'] >= med) & (x['slip'].fillna(0) <= 0.5))
    x['in_gate'] = gate
    return x


SAFE = ['content_age_days', 'impr_90d', 'avg_position']
FULL = SAFE + ['prior_trend', 'peak_ratio']
LEAN = [f for f in FULL if f != 'content_age_days']

cohort = build_cohort(D1)
ev = cohort[cohort['in_gate']].dropna(subset=FULL).copy()

print(f"cohort      {len(cohort):>7,} pages | {cohort['client_hash_id'].nunique():>2} clients "
      f"| decline rate {cohort['declined'].mean():.4f}")
print(f"gated pool  {int(cohort['in_gate'].sum()):>7,} pages "
      f"| {cohort.loc[cohort['in_gate'], 'client_hash_id'].nunique():>2} clients")
print(f"eval pool   {len(ev):>7,} pages | {ev['client_hash_id'].nunique():>2} clients "
      f"| decline rate {ev['declined'].mean():.4f}")
print("ML-07/ML-08 reported: cohort 202,073 / 53, pool 46,061 / 30, eval 45,095 / 30 / 0.5131")
print("  <- must match. If it does not, this notebook is describing a different cohort.")
print(f"features (shipped LEAN): {LEAN}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

cohort      202,073 pages | 53 clients | decline rate 0.4228
gated pool   46,061 pages | 30 clients
eval pool    45,095 pages | 30 clients | decline rate 0.5131
ML-07/ML-08 reported: cohort 202,073 / 53, pool 46,061 / 30, eval 45,095 / 30 / 0.5131
  <- must match. If it does not, this notebook is describing a different cohort.
features (shipped LEAN): ['impr_90d', 'avg_position', 'prior_trend', 'peak_ratio']


In [5]:
# Both helpers are copied verbatim from w07_action_playbook.ipynb so that every
# figure here is measured the way ML-10 measured it.

def queue_metrics(frame, score_col, k, ascending):
    """Per-client top-k. Returns pooled precision, macro precision, and clients scored.

    Pooled = total hits / total picks. Macro = mean of per-client precision, which is
    what FlyRank asks for: every client counts once regardless of size.
    """
    hits = picks = 0
    per = []
    for _, g in frame.groupby('client_hash_id'):
        kk = len(g) if k is None else min(k, len(g))
        if kk == 0:
            continue
        top = g.sort_values(score_col, ascending=ascending).head(kk)
        h = int(top['declined'].sum())
        hits += h
        picks += kk
        per.append(h / kk)
    return (hits / picks if picks else np.nan,
            float(np.mean(per)) if per else np.nan,
            len(per))


def lift_at(frame, target_col, k=100, n_splits=10):
    """Macro and pooled lift over random at the same K, on grouped splits."""
    frame = frame.copy()
    frame['declined'] = frame[target_col] < 0
    g = GroupShuffleSplit(n_splits=n_splits, test_size=0.2, random_state=SEED)
    P_, M_ = [], []
    for s, (itr, ite) in enumerate(g.split(frame, groups=frame['client_hash_id'])):
        tr, te = frame.iloc[itr], frame.iloc[ite]
        sc = StandardScaler().fit(tr[LEAN])
        m = Ridge(alpha=1.0).fit(sc.transform(tr[LEAN]), tr[target_col])
        rng = np.random.default_rng(s)
        t = te.assign(pred=m.predict(sc.transform(te[LEAN])), rand=rng.random(len(te)))
        pm, mm, _ = queue_metrics(t, 'pred', k, True)
        pr, mr, _ = queue_metrics(t, 'rand', k, False)
        P_.append(pm - pr)
        M_.append(mm - mr)
    return float(np.mean(P_)), float(np.mean(M_)), float(frame['declined'].mean())


print('helpers loaded: queue_metrics, lift_at')

helpers loaded: queue_metrics, lift_at


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**The split holds out whole clients.** `GroupShuffleSplit` on `client_hash_id`, ten splits, `test_size=0.2`, so no client's pages appear on both sides. This is justified empirically rather than asserted: the same features on a random *row* split score **+0.091** AUC higher on every one of ten seeds, and that gap is memorised client structure, not skill.

**The metric is rank agreement.** Spearman correlation between predicted and actual change is primary, because the task is ordering and rank correlation measures it directly without inventing a cut-off. Precision@K is reported on a **per-client** queue at K = 100 — pooled, with the base rate beside every figure. A continuous target has no positive class, so neither a base rate nor AUC is a property of the label any more; both return here at *evaluation*, where `target < 0` is applied.

The cell below re-derives the comparison on the same ten splits, and prints ML-08's reported figures beside it.

In [ ]:
# ascending=True  -> a LOW prediction means 'more likely to decline' (regressor on target)
# ascending=False -> a HIGH score means 'more likely to decline' (classifier on declined)
MODELS = [
    ('ridge FULL',    FULL, 'reg', True,  lambda: Ridge(alpha=1.0)),
    ('ridge LEAN',    LEAN, 'reg', True,  lambda: Ridge(alpha=1.0)),
    ('logistic FULL', FULL, 'clf', False, lambda: LogisticRegression(max_iter=2000, random_state=SEED)),
    ('gbm_reg FULL',  FULL, 'reg', True,  lambda: HistGradientBoostingRegressor(random_state=SEED)),
    ('gbm_clf FULL',  FULL, 'clf', False, lambda: HistGradientBoostingClassifier(random_state=SEED)),
    ('logistic SAFE', SAFE, 'clf', False, lambda: LogisticRegression(max_iter=2000, random_state=SEED)),
]

rows = []
for seed in range(10):
    itr, ite = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                    .split(ev, groups=ev['client_hash_id']))
    train, test = ev.iloc[itr], ev.iloc[ite]
    rng = np.random.default_rng(seed)
    base = test.assign(rand=rng.random(len(test)))
    bp, bm, _ = queue_metrics(base, 'rand', 100, False)
    base_rate = float(test['declined'].mean())
    for name, feats, kind, asc, make in MODELS:
        sc = StandardScaler().fit(train[feats])
        if kind == 'reg':
            m = make().fit(sc.transform(train[feats]), train['target'])
            pred = m.predict(sc.transform(test[feats]))
        else:
            m = make().fit(sc.transform(train[feats]), train['declined'].astype(int))
            pred = m.predict_proba(sc.transform(test[feats]))[:, 1]
        t = test.assign(pred=pred)
        p100, m100, _ = queue_metrics(t, 'pred', 100, asc)
        # Orient both kinds so a HIGHER value means a LESS likely decline, matching the
        # regressor's own direction. Left unoriented, the classifier's P(decline) flips
        # the rank correlation and inverts AUC, and the table reads as though ridge were
        # anti-predictive when it is ML-08's 0.7584.
        oriented = pred if kind == 'reg' else -pred
        rows.append({'model': name, 'seed': seed, 'pooled@100': p100, 'macro@100': m100,
                     'spearman': spearmanr(t['target'], oriented)[0],
                     'auc': roc_auc_score(t['declined'], -oriented),
                     'base_rate': base_rate, 'random@100': bp})

cmp = (pd.DataFrame(rows).groupby('model', sort=False)
       .agg(spearman=('spearman', 'mean'), pooled_100=('pooled@100', 'mean'),
            macro_100=('macro@100', 'mean'), auc=('auc', 'mean'),
            base_rate=('base_rate', 'mean'), random_100=('random@100', 'mean')))
cmp['lift_100'] = cmp['pooled_100'] - cmp['random_100']

# The baseline row is the rule's gate with pages in arbitrary order -- the bar to beat.
cmp.loc['baseline, gate + random'] = {
    'spearman': np.nan, 'pooled_100': cmp['random_100'].mean(),
    'macro_100': np.nan, 'auc': 0.5, 'base_rate': cmp['base_rate'].mean(),
    'random_100': cmp['random_100'].mean(), 'lift_100': 0.0}

print('model vs baseline -- ten client-grouped splits, per-client queues at K = 100')
print(f"pool base rate across the ten splits: {cmp['base_rate'].mean():.4f}")
print(cmp[['spearman', 'pooled_100', 'macro_100', 'auc', 'lift_100']]
      .to_string(float_format=lambda v: f'{v:.4f}'))

print('\nML-08 reported, for comparison:')
print('  ridge FULL spearman 0.5302 | P@100 0.7563 | AUC 0.7584')
print('  logistic FULL 0.5152 | 0.7500 | 0.7519')
print('  gbm_reg FULL 0.4945 | 0.7216 | 0.7380')
print('  gbm_clf FULL 0.4843 | 0.7298 | 0.7487')
print('  logistic SAFE 0.0553 | 0.5466 | 0.5436')
print('  baseline gate+random: P@100 0.5129')
print('  ML-09 ablation: ridge LEAN spearman 0.5546 | P@100 0.7612 | AUC 0.7693')
print('  <- must land close. A gap means the split construction has drifted from ML-08.')

In [7]:
# Where the model is wrong: the shape of the ordering, and where confidence ends.
drows = []
for s, (itr, ite) in enumerate(GroupShuffleSplit(n_splits=10, test_size=0.2,
                                                 random_state=SEED)
                                 .split(ev, groups=ev['client_hash_id'])):
    tr, te = ev.iloc[itr], ev.iloc[ite]
    sc = StandardScaler().fit(tr[LEAN])
    m = Ridge(alpha=1.0).fit(sc.transform(tr[LEAN]), tr['target'])
    t = te.assign(pred=m.predict(sc.transform(te[LEAN])))
    t = t.assign(decile=pd.qcut(t['pred'], 10, labels=False, duplicates='drop') + 1)
    for d, g in t.groupby('decile'):
        drows.append({'seed': s, 'decile': int(d), 'rate': float(g['declined'].mean()),
                      'clients': g['client_hash_id'].nunique()})

dec = (pd.DataFrame(drows).groupby('decile')
       .agg(decline_rate=('rate', 'mean')))
pool_rate = float(ev['declined'].mean())
dec['vs_pool'] = dec['decline_rate'] - pool_rate

print('actual decline rate by predicted decile (1 = model says most likely to fall)')
print(f'pool decline rate {pool_rate:.4f}')
print(dec.to_string(float_format=lambda v: f'{v:.4f}'))
print(f"\nmonotonic across all ten deciles: {bool((dec['decline_rate'].diff().dropna() < 0).all())}")
print(f"top decile {dec.loc[1, 'decline_rate']:.4f} | bottom decile "
      f"{dec.loc[dec.index.max(), 'decline_rate']:.4f} | "
      f"spread {dec.loc[1, 'decline_rate'] - dec.loc[dec.index.max(), 'decline_rate']:.4f}")
print(f"first decile no better than the pool base rate: {dec.index[dec['vs_pool'] <= 0].min()}")
print('ML-10 reported: 0.8923 -> 0.2081, spread 0.6842, crossover at decile 6  <- must match')

actual decline rate by predicted decile (1 = model says most likely to fall)
pool decline rate 0.5131
        decline_rate  vs_pool
decile                       
1             0.8923   0.3792
2             0.8089   0.2958
3             0.6455   0.1324
4             0.6167   0.1036
5             0.5312   0.0181
6             0.4375  -0.0756
7             0.3676  -0.1455
8             0.2782  -0.2349
9             0.2270  -0.2861
10            0.2081  -0.3050

monotonic across all ten deciles: True
top decile 0.8923 | bottom decile 0.2081 | spread 0.6842
first decile no better than the pool base rate: 6
ML-10 reported: 0.8923 -> 0.2081, spread 0.6842, crossover at decile 6  <- must match


## 5. Limitations

*What this work cannot claim.*

**It cannot predict magnitude.** R² is **0.0026**. The ordering is the product; the predicted value is not fit to display. Nothing here licenses "this page will lose 30% of its traffic".

**It makes no causal claim.** Nothing here shows that refreshing a page *causes* recovery — that would need an experiment this data cannot support. It says nothing about Google's ranking algorithm either.

**A decline rate is not a statement about a client.** A panel-level ratio — impressions per page-day, blind to every individual page — explains **95.7%** of the decline rate's movement (R² **0.9573**, correlation **−0.9784**) across fourteen weekly decision points. The decline rate climbs **0.3169 → 0.7873** while the panel ratio falls **1.7440 → 0.6574**, because the target compares a page's future against a baseline 30–90 days earlier, so an early decision point measures a rising future against a low baseline and a late one measures the reverse. Calling this a statement about the clients overstates it.

**The ranking, however, survives the confound** — which is why it is the part we ship. A panel-wide shift moves every page in a cohort together, so it changes the base rate without disturbing the order. The ranking is monotonic across all ten deciles, beats random on **13 of 13** forward weekly steps, and holds **45.8%** of its available ceiling (sd **5.4%**) while the pool's decline rate more than doubles. Over the same weeks raw P@100 lift falls **0.2559 → 0.1140** — which is the ceiling shrinking, not the model degrading.

**Staleness is not the problem it looks like.** An absolute lift measured at one date is not comparable to another, because a queue drawn from a pool declining at rate `b` cannot beat random by more than `1 − b`. Holding the test month fixed and moving only the training date, a 92-day-old model scores **0.1238** where a 31-day-old one scores **0.1242** — a gap of **0.0009**. A dashboard plotting raw lift would correlate **−0.9135** with the base rate and show a model in freefall that never degraded.

**Per-client modelling has no evidence behind it.** Clients genuinely differ, and per-client correlations are not noise. But both signals share `trend_recent_impr` with the target's own baseline, and an adversarial null replacing each page's future with a random window from its own history produces **−0.6225** where observation gives **−0.0306** — seven to twenty times more correlation from a null carrying no information. Grouping by client does not remove an artefact sitting inside each page. The per-client *queue* stands on capacity grounds; per-client *features* do not.

**Coverage, honestly.** At K = 100 the queue touches a small fraction of the gated pool, and **13 of 30** clients receive their entire eligible list at that budget — for them the queue is the gate, not a ranking. Recall is low because capacity binds, not because the ranking is weak.

**And the outer bound.** All of it is measured on one warehouse, at decision points from 2026-03-01 to 2026-06-01, on clients that survived the gate. Consecutive 90-day histories share 83 of 90 days, so the weekly walk measures week-to-week variation and cannot establish a trend.

**What must not be claimed:** that the model forecasts *how much* traffic will move; that acting on a queued page caused what followed; that a decline rate says anything about a client; or that any of this is validated beyond the decision points and window the warehouse allows.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

**Ship the ordering. Do not ship any number attached to it.** Across every test the model survived, what held was the rank; what moved was the level.

**Run the queue inside two segments, not as one list.** A page already below its own 90-day average declines **76.41%** of the time — a one-line filter needing no model. The ranking adds only **+0.0787** on top of it. On pages that have *not* started falling, where a filter leaves you at **40.69%**, it adds **+0.2186**. Normalised for available headroom the skill is comparable — **33.4%** against **36.9%** of ceiling — so this is not about where the model works but about where a rule already suffices. The global score sends **82.3%** of top-of-queue slots to the segment that needs it least, because `peak_ratio` dominates the ordering.

**Set K per client, and keep it small.** The advantage is concentrated at the top of each list, and **13 of 30** clients receive their entire eligible pool at K = 100 and are not being ranked at all — those clients should be told they are getting a filter.

**Stop at decile 5.** Decile 6 is the first no better than the pool base rate, and by decile 10 pages decline at **0.2081** against a pool rate of **0.5131** — the model is actively identifying them as healthy. Reviewing below the halfway mark is worse than sampling the pool at random.

**Apply a volume floor and route dead pages elsewhere.** The bottom volume quartile captures only **30.2%** of its ceiling, and **19.38%** of the top 100 per client are pages that reach zero — median `impr_90d` of **16** against **896** for the live pages beside them. A page at zero needs a redirect or a merge, which is irreversible in a way a content edit is not, and is not a decision this model has standing to inform.

**Choose the outcome window operationally.** Capture is flat from a 7-day horizon to a 90-day one — **53.7% - 59.2%**, mean **56.5%** — so the label need not match the rebuild cadence. A weekly rebuild reporting a 7-day outcome is as well supported as anything here.

**Monitor share of ceiling, not precision, and retrain rarely.** Alert when share of ceiling drops below **35.0%**, treating that floor as provisional — it was derived in-sample from the same thirteen steps it was tested on.

In [8]:
# The K sweep and the segment split, both copied from w07 so the figures below are
# measured the way ML-10 measured them.
KS = [10, 25, 50, 100, 250, 500, None]        # None = the client's whole gated pool
gss = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=SEED)

rows = []
for s, (itr, ite) in enumerate(gss.split(ev, groups=ev['client_hash_id'])):
    train, test = ev.iloc[itr], ev.iloc[ite]
    sc = StandardScaler().fit(train[LEAN])
    mdl = Ridge(alpha=1.0).fit(sc.transform(train[LEAN]), train['target'])
    rng = np.random.default_rng(s)
    t = test.assign(pred=mdl.predict(sc.transform(test[LEAN])), rand=rng.random(len(test)))
    for k in KS:
        pm, mm, nc = queue_metrics(t, 'pred', k, True)
        pr, mr, _ = queue_metrics(t, 'rand', k, False)
        rows.append({'K': 'all' if k is None else k, 'seed': s,
                     'pooled': pm, 'macro': mm,
                     'pooled_rand': pr, 'macro_rand': mr, 'clients': nc})

sw = (pd.DataFrame(rows).groupby('K', sort=False)
      .agg(pooled=('pooled', 'mean'), pooled_rand=('pooled_rand', 'mean'),
           macro=('macro', 'mean'), macro_rand=('macro_rand', 'mean')))
sw['pooled_lift'] = sw['pooled'] - sw['pooled_rand']
sw['macro_lift'] = sw['macro'] - sw['macro_rand']

print('K sweep -- ten grouped splits, per-client queues, lift over random at the same K')
print(sw[['pooled', 'pooled_rand', 'pooled_lift', 'macro', 'macro_lift']]
      .to_string(float_format=lambda v: f'{v:.4f}'))
print(f"\nbest pooled lift at K = {sw['pooled_lift'].idxmax()} "
      f"({sw['pooled_lift'].max():.4f})")
print(f"at K = all, lift is pooled {sw.loc['all', 'pooled_lift']:+.4f}  "
      f"<- must be ~0: the queue is the whole pool")
print('ML-10 reported: 0.3719 at K=10 down to 0.2492 at K=500, 0.0000 at K=all  <- must match')

# --- where the model earns its place: two segments, not one list -----------------
print('\n' + '=' * 68)
print('early warning or triage: split on the condition that drives 82.3% of the queue')
print('=' * 68)
for label, mask in [('already BELOW its 90-day average (peak_ratio < 1)', ev['peak_ratio'] < 1),
                    ('at or ABOVE its 90-day average (peak_ratio >= 1)', ev['peak_ratio'] >= 1)]:
    sub = ev[mask]
    ncl = sub['client_hash_id'].nunique()
    if len(sub) < 500 or ncl < 6:
        print(f'\n{label}: {len(sub):,} pages / {ncl} clients -- too few to split, skipped')
        continue
    pl, ml, br = lift_at(sub, 'target')
    print(f'\n{label}')
    print(f'  {len(sub):>6,} pages | {ncl} clients | decline rate {br:.4f}')
    print(f'  pooled lift {pl:+.4f} | macro lift {ml:+.4f}')
    print(f'  headroom {1 - br:.4f} -> captures {pl / (1 - br):.1%} of what is available')
print('\nML-10 reported: 13,410 pages / 28 clients / 0.7641 / +0.0787 / 33.4%')
print('                31,685 pages / 30 clients / 0.4069 / +0.2186 / 36.9%  <- must match')

K sweep -- ten grouped splits, per-client queues, lift over random at the same K
     pooled  pooled_rand  pooled_lift  macro  macro_lift
K                                                       
10   0.8766       0.5047       0.3719 0.8683      0.3483
25   0.8389       0.5139       0.3250 0.8259      0.2973
50   0.8157       0.5262       0.2895 0.7986      0.2603
100  0.8100       0.5344       0.2755 0.7775      0.2357
250  0.8127       0.5459       0.2667 0.7463      0.2064
500  0.8055       0.5563       0.2492 0.7176      0.1769
all  0.5013       0.5013       0.0000 0.5407      0.0000

best pooled lift at K = 10 (0.3719)
at K = all, lift is pooled +0.0000  <- must be ~0: the queue is the whole pool
ML-10 reported: 0.3719 at K=10 down to 0.2492 at K=500, 0.0000 at K=all  <- must match

early warning or triage: split on the condition that drives 82.3% of the queue

already BELOW its 90-day average (peak_ratio < 1)
  13,410 pages | 28 clients | decline rate 0.7641
  pooled lift +0.0787 

In [ ]:
# The forward walk in time: fourteen weekly decision points and thirteen forward steps.
# This is the evidence that the ordering survives a moving base rate -- and the reason
# the two measures below are TWO PANELS rather than one chart with two y-axes.
# Slow: build_cohort runs fourteen times. That is the cost of not caching a frame.
WEEKS = [str(d.date()) for d in pd.date_range('2026-03-01', periods=14, freq='7D')]
wk = {}
for w in WEEKS:
    c = build_cohort(w)
    wk[w] = c[c['in_gate']].dropna(subset=FULL).copy()

# Train on one week's cohort, score the NEXT week's. Forward only, never backwards:
# an in-week split trains and tests at the same decision point, which is not the walk
# ML-10 measured and not what the prose claims.
walk = []
for a, b in zip(WEEKS, WEEKS[1:]):
    tr, te = wk[a], wk[b]
    sc = StandardScaler().fit(tr[LEAN])
    m = Ridge(alpha=1.0).fit(sc.transform(tr[LEAN]), tr['target'])
    rng = np.random.default_rng(0)
    t = te.assign(pred=m.predict(sc.transform(te[LEAN])), rand=rng.random(len(te)))
    pm, mm, _ = queue_metrics(t, 'pred', 100, True)
    pr, mr, _ = queue_metrics(t, 'rand', 100, False)
    br = float(te['declined'].mean())
    walk.append({'week': b, 'base': br, 'pooled_lift': pm - pr, 'macro_lift': mm - mr,
                 'pct_ceiling': (pm - pr) / (1 - br)})
wf = pd.DataFrame(walk)
base_all = pd.Series([wk[w]['declined'].mean() for w in WEEKS], index=WEEKS)

pc = wf['pct_ceiling']
wins = int((wf['pooled_lift'] > 0).sum())
pl_mean, pl_sd = wf['pooled_lift'].mean(), wf['pooled_lift'].std()
pl_lo, pl_hi = wf['pooled_lift'].min(), wf['pooled_lift'].max()
# Two different correlations, and conflating them was a defect in this cell: raw
# lift tracks the base rate downwards, the ceiling share does not. Label each.
r_lift_base = float(wf['pooled_lift'].corr(wf['base']))
r_ceil_base = float(pc.corr(wf['base']))
# The floor is a rule, not ML-10's result. Hard-coding 0.35 froze a number that only
# equals mean - 2sd on ML-10's series, and fired on weeks this one calls fine.
lo = float(pc.mean() - 2 * pc.std())
below = int((pc < lo).sum())
print(f'weekly walk forward: {len(wf)} steps over {len(WEEKS)} decision points')
print(wf.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f'\nbase rate {base_all.iloc[0]:.4f} -> {base_all.iloc[-1]:.4f} '
      f'across the {len(WEEKS)} points')
print(f'pooled lift  mean {pl_mean:.4f} | sd {pl_sd:.4f} | range {pl_lo:.4f} - {pl_hi:.4f}')
print(f'share of ceiling: mean {pc.mean():.1%} | sd {pc.std():.1%} '
      f'| range {pc.min():.1%} - {pc.max():.1%}')
print(f'steps beating random: {wins} of {len(wf)}')
print(f'\ncorrelation with the test week base rate:')
print(f'  raw pooled lift   {r_lift_base:+.4f}   <- what a dashboard would plot')
print(f'  share of ceiling  {r_ceil_base:+.4f}   <- what actually reflects the model')
print(f'\nalert floor: share of ceiling below {lo:.1%} '
      f'(mean minus two sd of observed weekly variation)')
print(f'steps that would have fired: {below} of {len(wf)}')
print('ML-10 reported: base 0.3169 -> 0.7873, pooled lift range 0.1140 - 0.2559, '
      'share of ceiling mean 45.8% sd 5.4%, raw lift -0.9135 / ceiling +0.6575, '
      '13 of 13 beating random, 1 of 13 below the 35.0% floor  <- must match')

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Three figures, each carrying one message the prose already makes. They are written to `work/figures/` and the paper embeds them by relative path.

1. **Decline rate by predicted decile** — the ordering degrades gracefully and sets the stopping point at decile 5.
2. **Base rate and share of ceiling across the weekly walk** — the chart that would most like a second y-axis, and must not have one. Two stacked panels sharing an x-axis, because that split *is* the finding: the level moves, the ordering does not.
3. **Lift by K** — the advantage is concentrated at the top of each list.

Colours come from a validated categorical palette, not from taste: slot 1 blue `#2a78d6`, slot 2 orange `#eb6834`, on a `#fcfcfb` surface with `#0b0b0b` primary ink and a `#e1e0d9` hairline grid. Identity is never carried by colour alone — figures 1 and 3 are single-series so no legend is needed and the title names the measure, and figure 2's panels are titled individually.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# Palette: the reference instance's validated slots and chrome.
BLUE, ORANGE = '#2a78d6', '#eb6834'
SURFACE, INK, INK2, MUTED = '#fcfcfb', '#0b0b0b', '#52514e', '#898781'
GRID, BASELINE = '#e1e0d9', '#c3c2b7'

matplotlib.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'axes.edgecolor': BASELINE, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'text.color': INK,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.family': 'sans-serif', 'font.size': 10, 'figure.dpi': 140,
})

FIGDIR = Path('../figures')
FIGDIR.mkdir(parents=True, exist_ok=True)


def caption(fig, text):
    """One sentence under the chart, so the takeaway travels with the image."""
    fig.text(0.01, 0.005, text, ha='left', va='bottom', fontsize=9, color=INK2)


# --- 1. the ordering holds all the way down, and stops being useful at decile 6 ---
fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.bar(dec.index.astype(str), dec['decline_rate'], color=BLUE, width=0.68, linewidth=0)
ax.axhline(pool_rate, color=MUTED, linestyle='--', linewidth=1.4)
ax.annotate(f'pool base rate {pool_rate:.4f}', xy=(0.02, pool_rate),
            xycoords=('axes fraction', 'data'), va='bottom', fontsize=9, color=MUTED)
for i, v in enumerate(dec['decline_rate']):
    ax.text(i, v + 0.015, f'{v:.3f}', ha='center', fontsize=8, color=INK2)
ax.set_xlabel('predicted decile (1 = the model says most likely to fall)')
ax.set_ylabel('share that actually declined')
ax.set_ylim(0, 1.0)
ax.set_title('The ranking is monotonic across all ten deciles — and gives up at decile 6')
ax.grid(axis='x', visible=False)
caption(fig, f'Actual decline rate by predicted decile, ten client-grouped splits. Pool base rate '
             f'{pool_rate:.4f}. Decile 6 is the first no better than the pool; below it the model is '
             f'calling pages healthy.')
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(FIGDIR / 'deciles.png', bbox_inches='tight')
print('wrote', FIGDIR / 'deciles.png')

# --- 2. the level moves; the ordering does not. Two panels, NEVER two y-axes -----
fig, axes = plt.subplots(2, 1, figsize=(7.2, 4.8), sharex=True)
step = range(len(wf))
axes[0].plot(step, wf['base'], color=BLUE, linewidth=2, marker='o', markersize=4)
axes[0].set_ylabel('decline rate')
axes[0].set_title('The test week’s decline rate more than doubles across the thirteen forward steps')
axes[1].plot(step, wf['pct_ceiling'], color=ORANGE, linewidth=2, marker='o', markersize=4)
axes[1].set_ylabel('share of ceiling')
axes[1].set_xlabel('forward step, test week 2026-03-08 to 2026-05-31')
axes[1].set_title('While the model holds a stable share of the ceiling it is allowed')
axes[0].grid(axis='x', visible=False)
axes[1].grid(axis='x', visible=False)
caption(fig, 'Top: the decline rate of each step’s test week. Bottom: the model’s share of the '
             'ceiling each week allows. Separate panels, not two y-axes — the point is that the '
             'level moves while the ordering does not.')
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(FIGDIR / 'panel_vs_ceiling.png', bbox_inches='tight')
print('wrote', FIGDIR / 'panel_vs_ceiling.png')

# --- 3. the advantage lives at the top of each list ------------------------------
fig, ax = plt.subplots(figsize=(7.2, 3.8))
ks = [str(k) for k in sw.index]
ax.bar(ks, sw['pooled_lift'], color=BLUE, width=0.68, linewidth=0)
ax.axhline(0, color=BASELINE, linewidth=1.4)
for i, v in enumerate(sw['pooled_lift']):
    ax.text(i, v + 0.008, f'{v:.3f}', ha='center', fontsize=8, color=INK2)
ax.set_xlabel('per-client budget K (all = the client’s whole gated pool)')
ax.set_ylabel('pooled lift over random at the same K')
ax.set_title('The model’s advantage is concentrated at the top of each list')
ax.grid(axis='x', visible=False)
caption(fig, 'Pooled lift over random at the same K, ten client-grouped splits. Lift falls '
             'monotonically with budget and is exactly zero at K = all, where no ordering remains.')
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(FIGDIR / 'lift_by_k.png', bbox_inches='tight')
print('wrote', FIGDIR / 'lift_by_k.png')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.

> The boxes above are deliberately left unticked. Two of them — the run-all and the deployed paper — cannot be honestly ticked from inside this notebook, and the first is what `check_claims.py` then verifies: every number in the prose above has to be printed by a cell that actually ran.

## ML-12 — 5-minute demo outline

Five beats, about a minute each. One beat per section above, so the demo cannot drift from the notebook.

**0:00 — The decision.** A senior SEO specialist has more pages than review time and must choose which to open first. The unit is one content item at a decision date; the output is a ranked list where every row carries its reason. The model never edits a page.

**1:00 — The label that broke twice.** Started as three binary targets at an inherited ±20% cut; became one signed `asinh` difference. Two findings along the way: a 30-day window and a 45-day window disagree on 22.3% of pages, and the original decline flag was false *by construction* for already-declining pages — 89% of the gradient it showed was its own definition.

**2:00 — What survived every attack.** The ranking. Monotonic across all ten deciles, beating random on 13 of 13 forward weekly steps, holding 45.8% of its available ceiling (sd 5.4%) while the pool's decline rate more than doubled. Show figure 1, then figure 2.

**3:00 — Where it must not be trusted.** R² is 0.0026, so ship the rank and never the magnitude. A panel-level ratio explains 95.7% of the decline rate, so a decline rate is not a claim about a client. Nothing here is causal.

**4:00 — What I would do next.** Run the queue inside two segments, set K per client, stop at decile 5, route dead pages to a redirect decision instead. Then take questions.

## ML-12 — social-post cut

*One finding + one chart + one method sentence + link.*

> **A ranking model that beat random on 13 of 13 out-of-sample weeks — and the two things it must never be used for.**
>
> I ranked 45,095 content pages by likelihood of decline, holding out whole clients so the model never saw a client's pages on both sides of the split. The ranking held 45.8% of the ceiling each week allowed it while the pool's decline rate more than doubled — but R² was 0.0026, so the *order* is the product and the magnitude is not, and a panel-level ratio explains 95.7% of the decline rate, so it says nothing about any individual client.
>
> *Chart: `lift_by_k.png` — lift over random at the same K, falling monotonically from 0.3719 at K = 10 to exactly zero when the queue is the whole pool.*
>
> Built on the FlyRank ML Internship dataset. Full paper and notebooks: [link]

## ML-12 — employer-facing summary

*Three sentences: what I built · on what data · what it showed.*

I built a page-ranking model that tells a senior SEO specialist which content to review first, with a generated reason code on every row, and validated it by holding out entire clients rather than random rows. It is trained on FlyRank's gated internship warehouse — daily search-performance data for 202,073 content items across 53 clients, aggregated into one feature vector per page at a fixed decision date. The ranking beat a random ordering on 13 of 13 forward weekly steps and held 45.8% of the ceiling each week allowed it, but its magnitude was not predictable (R² 0.0026) and a panel-level trend explained 95.7% of the decline rate — so the deliverable is a reviewed, explained queue rather than an automated decision.